# Fine-tuning ya RAG: kab kaun-sa?
**AI Explained · Episode 17 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye. Do **toy** hain: ek RAG jaisa (sawal aane par notice padhta hai), ek fine-tune jaisa (ek baar seekhta hai, phir wahi yaad rakhta hai). Asli fine-tuning model ke parameters badalti hai; hamara toy sirf example yaad rakhta hai, idea samjhane ke liye. Notices aur cases hamne teaching ke liye banaye hain.

In [1]:
NOTICES = {
    "notice-1": "exam form ki last date "
                "10 december hai",
    "notice-2": "library subah 9 baje "
                "khulti hai",
    "notice-3": "placement talk friday "
                "ko hall mein hai",
}
print(len(NOTICES), "notices")

3 notices


In [2]:
def retrieve(q):   # sabse zyada same words
    qw = set(q.split())
    return max(NOTICES, key=lambda k:
        len(qw & set(NOTICES[k].split())))

def rag(q):        # padho, phir jawab
    k = retrieve(q)
    return NOTICES[k] + "  [source: " + k + "]"

Q = "exam form ki last date kya hai"
print("RAG:", rag(Q))

RAG: exam form ki last date 10 december hai  [source: notice-1]


In [3]:
def fine_tune(examples):
    # toy: asli FT parameters badalta hai
    return dict(examples)   # andar jam gaya

MODEL = fine_tune([(Q, "10 december")])

def ft(q):
    return MODEL.get(q, "pata nahi")

print("FT :", ft(Q))

FT : 10 december


In [4]:
# notice badla: nayi date
NOTICES["notice-1"] = ("exam form ki last "
                       "date 15 december hai")
print("RAG:", rag(Q))
print("FT :", ft(Q), " <- purani date")

RAG: exam form ki last date 15 december hai  [source: notice-1]
FT : 10 december  <- purani date


In [5]:
# FT ko dobara train karna padega
MODEL = fine_tune([(Q, "15 december")])
print("FT :", ft(Q), " <- retrain ke baad")

FT : 15 december  <- retrain ke baad


### Asli duniya mein
OpenAI ka *Optimizing LLM Accuracy* guide do raaste batata hai:

- **Context (RAG):** jab model ke paas jaankari nahi hai, wo purani hai, ya tumhari apni (proprietary) hai. *"This axis maximizes response accuracy."*
- **Model (fine-tuning):** jab format galat hai, tone ya style galat hai, ya reasoning har baar alag hai. *"This axis maximizes consistency of behavior."*
- **Shuruaat:** *"Prompt engineering is typically the best place to start."* Aur dono saath chal sakte hain: *"additive, not exclusive"*.

Google ka Machine Learning Crash Course: fine-tuning task ke examples par training hai, kabhi-kabhi *"just a few hundred or a few thousand training examples"* par; prompt engineering model ke parameters nahi badalti.

Research: Ovadia et al. (EMNLP 2024) ne naye facts sikhane mein RAG ko unsupervised fine-tuning se lagataar behtar paaya. Asli RAG paper (Lewis et al. 2020) ke mutabik, parameters mein rakhe knowledge ko update karna aur uska source dikhana mushkil hai.

In [6]:
def choose(c):
    if c["prompt_kaafi"]:
        return "PROMPT"
    need = []
    if c["facts"]:      # naye ya apne facts
        need.append("RAG")
    if c["aadat"]:      # format, tone, style
        need.append("FINE-TUNE")
    return " + ".join(need) or "PROMPT"

In [7]:
CASES = [  # (case, prompt_kaafi, facts, aadat)
  ("college notices ke sawal", 0, 1, 0),
  ("har reply company tone mein", 0, 0, 1),
  ("team docs + team format", 0, 1, 1),
  ("ek baar ka email", 1, 0, 0),
  ("HR policy ke sawal", 0, 1, 0),
  ("ticket ko fixed label do", 0, 0, 1),
]
from collections import Counter
tally = Counter()
for name, p, f, a in CASES:
    pick = choose({"prompt_kaafi": p,
                   "facts": f, "aadat": a})
    tally[pick] += 1
    print(f"{name:28s} -> {pick}")
print(len(CASES), "cases:", dict(tally))

college notices ke sawal     -> RAG
har reply company tone mein  -> FINE-TUNE
team docs + team format      -> RAG + FINE-TUNE
ek baar ka email             -> PROMPT
HR policy ke sawal           -> RAG
ticket ko fixed label do     -> FINE-TUNE
6 cases: {'RAG': 2, 'FINE-TUNE': 2, 'RAG + FINE-TUNE': 1, 'PROMPT': 1}


In [8]:
ANSWER = ("RAG gives the model the right "
  "documents at question time; the "
  "model does not change. Fine-tuning "
  "trains it on examples, so its "
  "format and tone stay consistent. "
  "Facts change or need a source: RAG. "
  "Fixed behaviour: fine-tuning. "
  "Try a prompt first.")
print(ANSWER)

RAG gives the model the right documents at question time; the model does not change. Fine-tuning trains it on examples, so its format and tone stay consistent. Facts change or need a source: RAG. Fixed behaviour: fine-tuning. Try a prompt first.


In [9]:
# TUMHARI BAARI: apna case daalo
mera = {"prompt_kaafi": 0,
        "facts": 1, "aadat": 1}
print(choose(mera))

RAG + FINE-TUNE
